In [ ]:
# fr/python-101/hard/09-temperature-tuning
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


Contrôler la créativité

Un modèle de langue à probabilités fixes produit toujours le même genre de sortie — il suit le corpus exactement. Mais parfois vous voulez un texte plus créatif et surprenant, et parfois vous voulez la sortie la plus prévisible et sûre. La **température** est le bouton qui contrôle ce compromis.

## Concepts clés

### Qu'est-ce que la température ?

La température est un nombre (généralement entre 0,1 et 2,0) qui met à l'échelle la distribution de probabilité du modèle avant l'échantillonnage :

- **Température basse** (par ex., 0,2) : Aiguise la distribution — le mot le plus probable devient encore plus probable, et les mots rares deviennent presque impossibles. La sortie est répétitive et prévisible.
- **Température 1,0** : Aucun changement — les probabilités d'origine sont utilisées telles quelles.
- **Température élevée** (par ex., 1,5) : Aplatit la distribution — tous les mots deviennent plus également probables. La sortie est plus aléatoire, créative et potentiellement insensée.

### Les mathématiques : mise à l'échelle des log-probabilités

La température fonctionne en divisant les log-probabilités par la valeur de température, puis en reconvertissant :


In [ ]:
import math

def apply_temperature(probabilities, temperature):
    """Apply temperature scaling to a probability distribution."""
    # Convert to log-probabilities
    log_probs = [math.log(p + 1e-10) for p in probabilities]  # add small epsilon to avoid log(0)

    # Scale by temperature
    scaled = [lp / temperature for lp in log_probs]

    # Convert back to probabilities (softmax-like)
    max_scaled = max(scaled)
    exp_scaled = [math.exp(s - max_scaled) for s in scaled]  # subtract max for numerical stability
    total = sum(exp_scaled)

    return [e / total for e in exp_scaled]


L'astuce `math.exp(s - max_scaled)` empêche le dépassement numérique — sans soustraire le maximum, les exponentielles pourraient être astronomiquement grandes.

### Exemple : distribution à trois mots


In [ ]:
words = ["cat", "dog", "bird"]
probs = [0.7, 0.2, 0.1]

# Low temperature: cat becomes even more dominant
cold = apply_temperature(probs, temperature=0.5)
print("Cold (0.5):", dict(zip(words, [f"{p:.3f}" for p in cold])))
# cat ≈ 0.876, dog ≈ 0.088, bird ≈ 0.036

# High temperature: more uniform distribution
hot = apply_temperature(probs, temperature=2.0)
print("Hot (2.0):", dict(zip(words, [f"{p:.3f}" for p in hot])))
# cat ≈ 0.524, dog ≈ 0.281, bird ≈ 0.195


### Intégration avec sample_next()

Modifiez la fonction d'échantillonnage pour accepter un paramètre de température :


In [ ]:
import random

def sample_next(model, current_word, temperature=1.0):
    if current_word not in model:
        return None

    followers = model[current_word]
    words = list(followers.keys())
    probs = list(followers.values())

    if temperature != 1.0:
        probs = apply_temperature(probs, temperature)

    return random.choices(words, weights=probs, k=1)[0]


À `temperature=1.0`, les probabilités d'origine sont utilisées sans modification. Les valeurs plus basses aiguisent ; les valeurs plus élevées aplatissent.

### Effets de la température sur la génération


In [ ]:
# Cold: repetitive, predictable
random.seed(42)
for _ in range(3):
    print(generate_text(model, "the", length=10, temperature=0.3))

# Hot: creative, surprising
random.seed(42)
for _ in range(3):
    print(generate_text(model, "the", length=10, temperature=1.5))


Avec une température basse, vous verrez les mêmes phrases courantes répétées. Avec une température élevée, vous obtiendrez des combinaisons de mots inhabituelles qui pourraient ne pas avoir de sens grammatical.

### Lignes directrices pratiques de température

| Température | Effet | Cas d'utilisation |
|-------------|--------|----------|
| 0,1–0,3 | Très déterministe | Reproduire un texte connu |
| 0,5–0,7 | Conservateur | Sortie factuelle et sûre |
| 0,8–1,0 | Équilibré | Génération à usage général |
| 1,0–1,5 | Créatif | Brainstorming, écriture créative |
| 1,5–2,0 | Très aléatoire | Sortie expérimentale et surprenante |

Pour un petit modèle de bigrammes, les températures au-dessus de 1,2 produisent souvent du charabia parce que le modèle n'a pas assez de contexte pour maintenir la cohérence quand le hasard est élevé.

## Essayez

Générez le même texte à trois températures différentes et comparez :


In [ ]:
random.seed(42)
for temp in [0.3, 1.0, 1.5]:
    print(f"\n[temperature={temp}]")
    for _ in range(3):
        print(f"  {generate_text(model, 'the', length=12, temperature=temp)}")


Quelle température produit la sortie la plus lisible ? Quelle est la plus surprenante ?

## Points clés

- La température met à l'échelle les distributions de probabilité : basse aiguise, élevée aplatit
- La température 1,0 signifie aucun changement des probabilités d'origine
- Implémentez-la en mettant à l'échelle les log-probabilités : `log_prob / temperature`
- Température basse (0,3-0,7) pour une sortie prévisible ; élevée (1,0+) pour une sortie créative

## Défi pratique

Écrivez une fonction `compare_temperatures(model, word, temps)` qui génère du texte à chaque température et affiche un tableau de comparaison :


In [ ]:
def compare_temperatures(model, word, temps=[0.3, 0.7, 1.0, 1.5], length=15):
    for temp in temps:
        random.seed(42)
        text = generate_text(model, word, length=length, temperature=temp)
        print(f"  T={temp:.1f}: {text}")


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
